# Lab 02 · RAG: ground answers in retrieved knowledge

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Retrieve architecture evidence before answering
**Goal:** Use RAG to retrieve and apply software architecture knowledge when answering a question.

**So far:** Lab 01 sent system and user prompts, reused an answer, and validated a response.

**New:** retrieval-augmented generation (RAG). We store short architecture notes, embed them, and retrieve the note that may answer a question. Check the answer against the retrieved note.

The local `all-MiniLM-L6-v2` model turns text into 384-number vectors. Chroma stores and searches those vectors on CPU; the first run downloads the model. No model key is needed for retrieval; generation uses the course client.

**At a glance**

![Lab 02 at a glance: Question, then Embed (MiniLM vectors), then Nearest architecture notes (Chroma), then close?, then Evidence-based reply (cites (R3)), then Citation check (cited ⊆ retrieved).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/02.svg)

## Think about the example

API composition asks services for data and joins their responses. CQRS maintains a separate read model for queries, which can lag behind writes. A citation identifies a source; it does not prove the answer follows from it.

**Concepts:** Database per service; API composition vs CQRS; Idempotent consumers and transactional outbox.

Use the scenario above to consider these questions before running the code:

1. Why is CQRS a better fit for searchable order history than a large in-memory API join?
2. Does citing a valid source ID prove that the source supports an answer?

## The idea in context

Retrieval-augmented generation puts selected external evidence into a prompt. Embeddings find semantic neighbours; neighbours still need relevance checks.

Citing a valid source ID does not by itself prove factual support.

| Key term | Meaning |
| --- | --- |
| **Context window** | The maximum amount of text a model can read in one call. |
| **Embedding** | A vector of numbers that represents the meaning of a text. |
| **Vector database** | A store that finds the vectors nearest to a query vector. |
| **Grounded answer** | An answer supported by, and citing, the retrieved sources. |

## Pattern · Retrieval-Augmented Generation

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Useful answers often depend on private, recent, or domain-specific material that the model's weights do not reliably hold. |
| **Problem** | Without evidence, a model may hallucinate, miss local facts, or answer from outdated knowledge. |
| **Solution** | Search a knowledge base for relevant pieces, place them beside the question, and instruct the model to ground its answer in them. |
| **Use when** | Answers depend on a document collection; citations or provenance matter; knowledge changes independently of the model. |
| **Trade-off** | Answer quality now depends on retrieval quality, chunking, and the context budget. |
| **Flow** | Question → Retrieve → Augment → Grounded answer |
| **Quality attributes** | Benefits: Reliability, Data freshness, Modifiability, Cost, Privacy. Liabilities: Latency, Maintainability, Reliability, Data freshness. |
| **In this lab** | Embeddings and Chroma retrieve architecture notes with source IDs. A threshold supports abstention. Code checks citation validity; students inspect whether the source actually supports the claim. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Store architecture notes with source IDs
`notes` maps each source ID to a short architecture guideline. For example, the data-ownership note says that another service should use an API or event rather than access private tables. The IDs let us retrieve and cite those guidelines.

In [ ]:
notes = {
    "R1": "Database per service: each service owns private data; other services use APIs or events, not its tables.",
    "R2": "API composition joins a few services in the caller for a small lookup by id; large searches are a poor fit.",
    "R3": "CQRS maintains a separate read model for searchable order history; the view can lag behind writes.",
    "R4": "At-least-once delivery can repeat an event; store the processed eventId and the business effect in the same local DB transaction.",
    "R5": "Transactional outbox commits business state and publication intent in one local transaction; relay retries may duplicate.",
}
question = "Support needs searchable order history across services. Which data pattern fits, and what is its cost?"
for sid, text in notes.items():
    print(sid, text)

## Step 2 · Compare semantic vectors
Cosine similarity compares the directions of two vectors. It is a retrieval heuristic, not proof that a note answers the question.

In [ ]:
import numpy as np
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed = DefaultEmbeddingFunction()
question_vector, cqrs_vector, ownership_vector = embed([question, notes["R3"], notes["R1"]])

def cosine(first, second):
    dot_product = np.dot(first, second)
    vector_lengths = np.linalg.norm(first) * np.linalg.norm(second)
    return float(dot_product / vector_lengths)

print("Vector size:", len(question_vector))
print("Question vs CQRS:", round(cosine(question_vector, cqrs_vector), 2))
print("Question vs ownership:", round(cosine(question_vector, ownership_vector), 2))

## Step 3 · Search a vector collection
A smaller cosine distance means a nearer match. We retain each note ID as its source identifier. This is an in-memory teaching store, not persistent course infrastructure. `upsert` inserts a note or updates an existing ID, so rerunning the cell is safe. Chroma accepts several questions at once, so its results are lists of lists; `[0]` selects the results for our single question. `zip` pairs each source ID with its distance.

In [ ]:
import chromadb
from chromadb.config import Settings

database = chromadb.Client(Settings(anonymized_telemetry=False))
collection = database.get_or_create_collection(
    "lecture-notes", embedding_function=embed,
    configuration={"hnsw": {"space": "cosine"}},
)
collection.upsert(ids=list(notes), documents=list(notes.values()))
found = collection.query(query_texts=[question], n_results=3)
source_ids = found["ids"][0]
distances = found["distances"][0]
for source_id, distance in zip(source_ids, distances):
    print(source_id, round(distance, 2))

## Step 4 · Retrieve or abstain
The threshold is a small classroom heuristic that must be tuned on real questions. Unrelated questions should not reach the generator. Inspect distances before trusting it.

In [ ]:
def retrieve(question, max_distance=0.7):
    found = collection.query(query_texts=[question], n_results=1)
    nearest_id = found["ids"][0][0]
    nearest_distance = found["distances"][0][0]
    if nearest_distance > max_distance:
        return {}  # no suitable evidence
    return {nearest_id: notes[nearest_id]}

questions = [
    question,
    "May Reporting read the private tables owned by Order?",
    "What should we eat for lunch?",
]
for current_question in questions:
    hits = retrieve(current_question)
    print(list(hits), current_question)

## Step 5 · Answer with a citation, then check support
The code checks **citation validity**: a cited ID must have been retrieved. That alone does not prove the source supports the claim. Students must inspect that semantic support and the stated trade-off. `re.findall` extracts IDs such as R3 from the answer. A set removes repeated citations; `issubset` checks that every cited ID belongs to the retrieved evidence. `all(valid)` requires every answered question to pass that check.

In [ ]:
import re

scripted = [
    "Use a CQRS read model for searchable history; it may lag behind writes [R3].",
    "No: Order owns its private data; Reporting must use an API or events [R1].",
    "",
]
valid = []
for current_question, example in zip(questions, scripted):
    hits = retrieve(current_question)
    if not hits:
        print("No matching architecture note; no model call:", current_question)
        continue
    evidence_lines = []
    for source_id, text in hits.items():
        evidence_lines.append(f"[{source_id}] {text}")
    evidence = "\n".join(evidence_lines)
    prompt = f"Use only these notes:\n{evidence}\nQuestion: {current_question}\nReply briefly and cite the source ID."
    answer = ask(prompt, example=example)
    cited_ids = set(re.findall(r"R\d+", answer))
    citation_valid = bool(cited_ids) and cited_ids.issubset(hits)
    valid.append(citation_valid)
    print(answer, "| citation valid:", citation_valid)
assert all(valid), "Cite only retrieved source IDs."

## What you should see
Searchable history retrieves R3; private-table access retrieves R1. The lunch question abstains. Answers cite their retrieved notes. Distances can vary with package/model versions.

## Try it
1. A low-traffic order-details page needs a lookup by ID from three services. Which note suggests API composition, and what is its limitation?
2. An order commits just before its producer crashes. Inspect the transactional outbox note: how can the system preserve the intent to publish, and why is this not one transaction across services?
3. Put an incorrect answer beside a valid citation. Why does the code accept the citation while a human should reject the answer?

## Takeaway
Retrieval selects evidence; generation uses it; citation validity and factual support are different checks.